In [1]:
from pathlib import Path
import pandas as pd

RES = Path.cwd().parent / 'results'
df = pd.read_csv(RES / 'metrics.csv')
df

,model,split,seed,elapsed_s,train_rmse,train_mae,train_r2,train_n,val_rmse,val_mae,val_r2,val_n,test_rmse,test_mae,test_r2,test_n
0,A,random,42,173.4,0.174106,0.117611,0.979962,80786,0.188473,0.128083,0.976264,10098,0.201292,0.130473,0.972117,10099
1,A,textrap,42,133.3,0.186152,0.126839,0.976815,67902,0.197350,0.136525,0.973192,7385,0.240926,0.173192,0.959589,25696
2,A,coldsol,42,46.3,0.253809,0.176372,0.957641,81610,0.808241,0.597208,0.527351,8937,0.870930,0.656302,0.489160,10436
3,A,coldpair,42,113.1,0.179009,0.124967,0.978693,80806,0.495371,0.327254,0.834330,10187,0.555691,0.365603,0.799356,9990
4,A,coldsolv,42,43.4,0.280745,0.199927,0.950323,77981,0.516715,0.391342,0.776158,11262,0.448639,0.312039,0.829474,11740
5,D,random,42,101.9,0.202573,0.140229,0.972874,80786,0.214173,0.149514,0.969349,10098,0.230507,0.153379,0.963436,10099
6,D,textrap,42,97.0,0.196001,0.134375,0.974297,67902,0.209375,0.144292,0.969826,7385,0.252424,0.182372,0.955640,25696
7,D,coldsol,42,33.2,0.272855,0.188297,0.951045,81610,0.811499,0.602250,0.523533,8937,0.892573,0.681364,0.463457,10436
8,D,coldpair,42,65.4,0.202819,0.141331,0.972648,80806,0.494010,0.326115,0.835239,10187,0.551354,0.358656,0.802476,9990
9,D,coldsolv,42,43.4,0.268264,0.185350,0.954641,77981,0.547422,0.422726,0.748762,11262,0.438357,0.297941,0.837201,11740


In [2]:
ref = df.query("split == 'random'").set_index('model')['test_rmse'].to_dict()
ref

{'A': 0.2012915164232254, 'D': 0.2305071502923965, 'B': 0.2781128287315368}

In [3]:
SPLIT_ORDER = ['random', 'textrap', 'coldsol', 'coldpair', 'coldsolv']
SPLIT_LABEL = {'random': 'random', 'textrap': 'T-extrap',
               'coldsol': 'cold-solute', 'coldpair': 'cold-pair', 'coldsolv': 'cold-solvent'}
MODEL_ORDER = ['A', 'D', 'B']
MODEL_LABEL = {'A': 'A direct', 'D': 'D 1/T control', 'B': "B Van't Hoff"}

rows = []
for m in MODEL_ORDER:
    for s in SPLIT_ORDER:
        r = df.query(f"model == '{m}' and split == '{s}'").iloc[0]
        rows.append({
            'model': MODEL_LABEL[m],
            'split': SPLIT_LABEL[s],
            'test_rmse': round(r['test_rmse'], 3),
            'test_r2':   round(r['test_r2'], 3),
            'delta_rmse': round(r['test_rmse'] - ref[m], 3) if s != 'random' else 0.0,
        })
summary = pd.DataFrame(rows)
summary

,model,split,test_rmse,test_r2,delta_rmse
0,A direct,random,0.201,0.972,0.000
1,A direct,T-extrap,0.241,0.960,0.040
2,A direct,cold-solute,0.871,0.489,0.670
3,A direct,cold-pair,0.556,0.799,0.354
4,A direct,cold-solvent,0.449,0.829,0.247
5,D 1/T control,random,0.231,0.963,0.000
6,D 1/T control,T-extrap,0.252,0.956,0.022
7,D 1/T control,cold-solute,0.893,0.463,0.662
8,D 1/T control,cold-pair,0.551,0.802,0.321
9,D 1/T control,cold-solvent,0.438,0.837,0.208


In [4]:
for s in SPLIT_ORDER:
    sub = df.query(f"split == '{s}'").sort_values('test_rmse')
    order = ' -> '.join(f"{r['model']} ({r['test_rmse']:.3f})" for _, r in sub.iterrows())
    print(f'{SPLIT_LABEL[s]:14s}  absolute best-to-worst: {order}')

random          absolute best-to-worst: A (0.201) -> D (0.231) -> B (0.278)
T-extrap        absolute best-to-worst: A (0.241) -> D (0.252) -> B (0.328)
cold-solute     absolute best-to-worst: A (0.871) -> D (0.893) -> B (0.893)
cold-pair       absolute best-to-worst: D (0.551) -> A (0.556) -> B (0.566)
cold-solvent    absolute best-to-worst: D (0.438) -> A (0.449) -> B (0.524)


In [5]:
for s in SPLIT_ORDER:
    if s == 'random':
        continue
    sub = df.query(f"split == '{s}'").copy()
    sub['delta'] = sub.apply(lambda r: r['test_rmse'] - ref[r['model']], axis=1)
    sub = sub.sort_values('delta')
    order = ' -> '.join(f"{r['model']} (Δ={r['delta']:+.3f})" for _, r in sub.iterrows())
    print(f'{SPLIT_LABEL[s]:14s}  smallest ΔRMSE -> largest: {order}')

T-extrap        smallest ΔRMSE -> largest: D (Δ=+0.022) -> A (Δ=+0.040) -> B (Δ=+0.050)
cold-solute     smallest ΔRMSE -> largest: B (Δ=+0.615) -> D (Δ=+0.662) -> A (Δ=+0.670)
cold-pair       smallest ΔRMSE -> largest: B (Δ=+0.288) -> D (Δ=+0.321) -> A (Δ=+0.354)
cold-solvent    smallest ΔRMSE -> largest: D (Δ=+0.208) -> B (Δ=+0.246) -> A (Δ=+0.247)


In [6]:
lines = []
w = lines.append
w('# Stage-2 comparison — A vs D vs B on 5 splits (seed 42)')
w('')
w('| Model | Split | Test RMSE | Test R² | ΔRMSE vs random |')
w('|---|---|---:|---:|---:|')
for m in MODEL_ORDER:
    for s in SPLIT_ORDER:
        r = df.query(f"model == '{m}' and split == '{s}'").iloc[0]
        delta = r['test_rmse'] - ref[m]
        d_str = f'{delta:+.3f}' if s != 'random' else '  0.000'
        w(f'| {MODEL_LABEL[m]} | {SPLIT_LABEL[s]} | {r["test_rmse"]:.3f} | '
          f'{r["test_r2"]:.3f} | {d_str} |')
report = '\n'.join(lines)
(RES / 'comparison_stage2.md').write_text(report)
print('wrote', RES / 'comparison_stage2.md')
print()
print(report)

wrote /home/sg182/ML_Projects/BigSolDB-T/results/comparison_stage2.md

# Stage-2 comparison — A vs D vs B on 5 splits (seed 42)

| Model | Split | Test RMSE | Test R² | ΔRMSE vs random |
|---|---|---:|---:|---:|
| A direct | random | 0.201 | 0.972 |   0.000 |
| A direct | T-extrap | 0.241 | 0.960 | +0.040 |
| A direct | cold-solute | 0.871 | 0.489 | +0.670 |
| A direct | cold-pair | 0.556 | 0.799 | +0.354 |
| A direct | cold-solvent | 0.449 | 0.829 | +0.247 |
| D 1/T control | random | 0.231 | 0.963 |   0.000 |
| D 1/T control | T-extrap | 0.252 | 0.956 | +0.022 |
| D 1/T control | cold-solute | 0.893 | 0.463 | +0.662 |
| D 1/T control | cold-pair | 0.551 | 0.802 | +0.321 |
| D 1/T control | cold-solvent | 0.438 | 0.837 | +0.208 |
| B Van't Hoff | random | 0.278 | 0.947 |   0.000 |
| B Van't Hoff | T-extrap | 0.328 | 0.925 | +0.050 |
| B Van't Hoff | cold-solute | 0.893 | 0.463 | +0.615 |
| B Van't Hoff | cold-pair | 0.566 | 0.792 | +0.288 |
| B Van't Hoff | cold-solvent | 0.524 | 0.76